In [ ]:
import pandas as pd
import os
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
import string
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)

nlp = spacy.load("en_core_web_lg")

In [ ]:
df = pd.read_csv('../data/dcInbox/dcinbox_export_116.csv')
unnamed_cols = df.columns.str.contains('^Unnamed')
df = df.loc[:, ~unnamed_cols].copy()
df = df[pd.to_numeric(df['Unix Timestamp'], errors='coerce').notna()].copy()
df['datetime'] = pd.to_datetime(df['Unix Timestamp'], unit='ms')

# Sort chronologically
df = df.sort_values('datetime').reset_index(drop=True)


In [ ]:
# print min and max dates
print("Min date:", df['datetime'].min())
print("Max date:", df['datetime'].max())

In [ ]:
## Preprocessing functions to apply to catalogs' aggregated course descriptions
def remove_double_spaces(text):
    if not text:
        return ""
    return " ".join(text.split())


def remove_paragraphs_dash(text):
    return re.sub(r"-\n", "", text)


def remove_paragraphs(text):
    return re.sub("\n", " ", text)

In [ ]:
LEXICON = {    
    'Election Integrity Terms': [
        'election integrity',
        'voter fraud',
        'voter id',
        'election reform',
        'voter verification',
        'ballot security',
        'election security',
        'mail-in voting concerns',
        'voter suppression',
        'election audits',
        'American elections',
        'proof of citizenship',
        'election interference',
        'foreign interference',
        'paper ballots',
        'voting machines',
        'cybersecurity',
        'election theft',
    ],
}

In [ ]:
content = (
    df['Body']
    .apply(remove_double_spaces)
    .apply(remove_paragraphs)
    .apply(remove_paragraphs_dash)
    .to_numpy()
)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
X = model.encode(content, show_progress_bar=True)

In [ ]:
print(f"content.shape: {content.shape}")
print(f"X.shape: {X.shape}")


In [ ]:
election_integrity_terms = LEXICON['Election Integrity Terms']
election_integrity_qv = model.encode(election_integrity_terms, show_progress_bar=True)
similarity_matrix = cosine_similarity(election_integrity_qv, X)

In [ ]:
threshold = 0.55

# Convert numpy array to DataFrame
similarity_matrix_df = pd.DataFrame(
    data=similarity_matrix.T,  # Transpose so rows=emails, columns=terms
    columns=election_integrity_terms
)
similarity_matrix_df['ID'] = df['ID'].values

# Find rows where ANY term has similarity > threshold
mask = (similarity_matrix_df[election_integrity_terms] > threshold).any(axis=1)
high_similarity_ids = similarity_matrix_df[mask]['ID'].values

# Create new dataframe with all original columns plus similarity info
filtered_df = df[df['ID'].isin(high_similarity_ids)].copy()

# Optionally, add columns showing which terms matched and their scores
for term in election_integrity_terms:
    # Merge the similarity scores back to the filtered dataframe
    term_scores = similarity_matrix_df[['ID', term]].rename(columns={term: f'{term}_score'})
    filtered_df = filtered_df.merge(term_scores, on='ID', how='left')

# Filter out score columns where the value is <= threshold (optional cleanup)
score_columns = [col for col in filtered_df.columns if col.endswith('_score')]
for col in score_columns:
    filtered_df.loc[filtered_df[col] <= threshold, col] = None

print(f"Found {len(filtered_df)} emails with similarity scores > {threshold}")
print(f"\nDataframe shape: {filtered_df.shape}")

In [ ]:
filtered_df.to_csv('../data/election_integrity_sample_emails.csv')